In [27]:
import os

# 1. Đường dẫn file nén (Đảm bảo bạn đã upload file này lên Colab)
FILE_PATH = "/content/en_ewt-ud.jsonl.tar.gz"

# 2. Giải nén
print(f"Đang giải nén {FILE_PATH}...")
try:
    # Cờ -xzvf: eXtract, gZip, Verbose (hiện chi tiết), File
    !tar -xzvf {FILE_PATH}
    print("Giải nén thành công!")
except Exception as e:
    print(f"Lỗi: {e}. Hãy kiểm tra lại xem đã upload file chưa.")

# 3. Kiểm tra xem giải nén ra cái gì
print("\nDanh sách các file sau khi giải nén:")
!ls -R
# Bạn cần nhìn output của lệnh này để biết tên folder chứa dữ liệu (ví dụ: en_ewt-ud/...)

Đang giải nén /content/en_ewt-ud.jsonl.tar.gz...
en_ewt-ud-dev.jsonl
en_ewt-ud-test.jsonl
en_ewt-ud-train.jsonl
Giải nén thành công!

Danh sách các file sau khi giải nén:
.:
data		     en_ewt-ud.jsonl.tar.gz  en_ewt-ud-train.jsonl  hwu.tar.gz
en_ewt-ud-dev.jsonl  en_ewt-ud-test.jsonl    hwu		    sample_data

./data:
UD_English-EWT

./data/UD_English-EWT:

./hwu:
categories.json  test.csv  train_10.csv  train_5.csv  train.csv  val.csv

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [37]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import os

# 1. Chọn thiết bị (GPU nếu có, không thì CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

# 2. Định nghĩa các Token đặc biệt
# PAD: Dùng để đệm cho câu ngắn bằng câu dài
# UNK: Dùng cho các từ lạ không có trong từ điển
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

Đang sử dụng thiết bị: cpu


In [39]:
# --- HÀM ĐỌC FILE JSONL ---
def load_jsonl(file_path):
    sentences = []
    print(f"Đang đọc file: {file_path}")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    # Lấy dữ liệu theo đúng key bạn cung cấp
                    tokens = data.get('words')
                    tags = data.get('tags')

                    if tokens and tags:
                        # Kiểm tra độ dài khớp nhau
                        if len(tokens) == len(tags):
                            # Ghép thành cặp [(word, tag), (word, tag)...]
                            sentences.append(list(zip(tokens, tags)))
                except json.JSONDecodeError:
                    continue # Bỏ qua dòng lỗi
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file {file_path}. Hãy kiểm tra lại đường dẫn.")
        return []

    return sentences

# --- HÀM XÂY DỰNG TỪ ĐIỂN ---
def build_vocab(sentences):
    print("Đang xây dựng từ điển...")
    word_counter = Counter()
    tag_set = set()

    for sent in sentences:
        for word, tag in sent:
            word_counter[word] += 1
            tag_set.add(tag)

    # 1. Tạo từ điển Word -> Index
    # Index 0 là PAD, 1 là UNK
    word_to_ix = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    # Chỉ thêm các từ xuất hiện ít nhất 1 lần
    for word, _ in word_counter.most_common():
        word_to_ix[word] = len(word_to_ix)

    # 2. Tạo từ điển Tag -> Index
    # Index 0 là PAD (để khi tính Loss ta bỏ qua nó)
    tag_to_ix = {PAD_TOKEN: PAD_IDX}
    for tag in sorted(list(tag_set)):
        tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix

# --- THỰC THI BƯỚC 2 ---
# Thay đường dẫn tới file đã giải nén của bạn
# Ví dụ: nếu giải nén ra thư mục 'en_ewt-ud', tên file thường là 'en_ewt-ud-train.jsonl'
# Bạn hãy kiểm tra tên file thật bằng lệnh !ls en_ewt-ud trong 1 cell khác

train_path = 'en_ewt-ud-train.jsonl' # <--- SỬA ĐƯỜNG DẪN Ở ĐÂY NẾU KHÁC
dev_path = 'en_ewt-ud-dev.jsonl'     # <--- SỬA ĐƯỜNG DẪN Ở ĐÂY NẾU KHÁC

# Load dữ liệu
train_data = load_jsonl(train_path)
dev_data = load_jsonl(dev_path)

if len(train_data) > 0:
    # Xây dựng từ điển từ tập Train
    word_to_ix, tag_to_ix = build_vocab(train_data)

    print(f"\n--- THÔNG TIN DỮ LIỆU ---")
    print(f"Số câu Train: {len(train_data)}")
    print(f"Số câu Dev:   {len(dev_data)}")
    print(f"Kích thước bộ từ vựng (Vocab): {len(word_to_ix)}")
    print(f"Số lượng nhãn (Tags): {len(tag_to_ix)}")
    print(f"Danh sách nhãn: {list(tag_to_ix.keys())}")
else:
    print("Dữ liệu rỗng! Vui lòng kiểm tra lại đường dẫn file.")

Đang đọc file: en_ewt-ud-train.jsonl
Đang đọc file: en_ewt-ud-dev.jsonl
Đang xây dựng từ điển...

--- THÔNG TIN DỮ LIỆU ---
Số câu Train: 12544
Số câu Dev:   2001
Kích thước bộ từ vựng (Vocab): 19675
Số lượng nhãn (Tags): 18
Danh sách nhãn: ['<PAD>', 'ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [40]:
# --- LỚP DATASET ---
class POSDataset(Dataset):
    def __init__(self, data, word_to_ix, tag_to_ix):
        self.data = data
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.data[idx]
        # Chuyển chữ -> số (dùng UNK nếu từ lạ)
        word_indices = [self.word_to_ix.get(word, UNK_IDX) for word, tag in sentence]
        tag_indices = [self.tag_to_ix[tag] for word, tag in sentence]

        return torch.tensor(word_indices), torch.tensor(tag_indices)

# --- HÀM COLLATE (Xử lý Padding) ---
def collate_fn(batch):
    sentences, tags = zip(*batch)

    # Pad sequences để độ dài bằng nhau trong 1 batch (thêm số 0 vào cuối câu ngắn)
    # batch_first=True giúp output có dạng (Batch, Seq_Len)
    padded_sentences = pad_sequence(sentences, batch_first=True, padding_value=PAD_IDX)
    padded_tags = pad_sequence(tags, batch_first=True, padding_value=PAD_IDX)

    return padded_sentences, padded_tags

# --- TẠO DATALOADER ---
BATCH_SIZE = 32

if len(train_data) > 0:
    train_dataset = POSDataset(train_data, word_to_ix, tag_to_ix)
    dev_dataset = POSDataset(dev_data, word_to_ix, tag_to_ix)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    print("Đã tạo DataLoader thành công.")

Đã tạo DataLoader thành công.


In [41]:
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx):
        super().__init__()

        # 1. Embedding Layer: Chuyển index từ -> vector đặc trưng
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # 2. RNN Layer: Xử lý chuỗi
        # batch_first=True là bắt buộc để khớp với DataLoader ở trên
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # 3. Linear Layer: Chiếu output của RNN ra số lượng nhãn (Tags)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x shape: [batch, seq_len]

        # embedded shape: [batch, seq_len, emb_dim]
        embedded = self.embedding(x)

        # rnn_out shape: [batch, seq_len, hid_dim]
        # Ta lấy toàn bộ sequence output vì đây là bài toán many-to-many
        rnn_out, _ = self.rnn(embedded)

        # predictions shape: [batch, seq_len, output_dim]
        predictions = self.fc(rnn_out)

        return predictions

In [42]:
# --- HÀM HUẤN LUYỆN (TRAIN) ---
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for batch in iterator:
        text, tags = batch
        # Đẩy dữ liệu lên GPU (nếu có)
        text = text.to(device)
        tags = tags.to(device)

        optimizer.zero_grad()

        # Forward pass
        predictions = model(text) # [batch, seq_len, num_tags]

        # Reshape dữ liệu để tính CrossEntropyLoss
        # Gộp batch và seq_len thành 1 chiều dài
        predictions = predictions.view(-1, predictions.shape[-1])
        tags = tags.view(-1)

        # Tính Loss (sẽ tự động bỏ qua các tag là PAD_IDX)
        loss = criterion(predictions, tags)

        # Backward pass
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

# --- HÀM ĐÁNH GIÁ (EVALUATE) ---
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    total_tokens = 0
    correct_tokens = 0

    with torch.no_grad():
        for batch in iterator:
            text, tags = batch
            text = text.to(device)
            tags = tags.to(device)

            predictions = model(text)

            # Tính Loss
            predictions_flat = predictions.view(-1, predictions.shape[-1])
            tags_flat = tags.view(-1)
            loss = criterion(predictions_flat, tags_flat)
            epoch_loss += loss.item()

            # Tính Accuracy
            # Lấy nhãn có xác suất cao nhất
            preds = torch.argmax(predictions, dim=-1) # [batch, seq_len]

            # Tạo mask: True ở những chỗ không phải PAD, False ở chỗ PAD
            mask = (tags != PAD_IDX)

            # Chỉ đếm đúng sai ở những chỗ mask == True
            correct = (preds == tags) & mask

            correct_tokens += correct.sum().item()
            total_tokens += mask.sum().item()

    return epoch_loss / len(iterator), correct_tokens / total_tokens

In [43]:
# --- THAM SỐ HUẤN LUYỆN ---
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = len(tag_to_ix) # Số lượng nhãn đầu ra
INPUT_DIM = len(word_to_ix) # Kích thước từ điển đầu vào
N_EPOCHS = 10
LEARNING_RATE = 0.01 # Tốc độ học

# 1. Khởi tạo Model
model = SimpleRNNForTokenClassification(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, PAD_IDX)
model = model.to(device) # Đẩy model lên GPU

# 2. Khởi tạo Optimizer và Loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# QUAN TRỌNG: ignore_index=PAD_IDX giúp Loss Function bỏ qua số 0 (Padding)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# 3. Vòng lặp Huấn luyện
print(f"Bắt đầu huấn luyện trên {device}...")

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, dev_loader, criterion)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

# 4. Dự đoán thử một câu (Optional)
def predict_sentence(model, sentence, word_to_ix, tag_to_ix, device):
    model.eval()
    tokens = sentence.split()
    indexes = [word_to_ix.get(token, UNK_IDX) for token in tokens]
    tensor = torch.LongTensor(indexes).unsqueeze(0).to(device) # Thêm batch dim [1, seq_len]

    with torch.no_grad():
        prediction = model(tensor)
        max_preds = prediction.argmax(dim=-1).squeeze(0) # [seq_len]

    # Map từ index ngược lại tag
    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    predicted_tags = [ix_to_tag[ix.item()] for ix in max_preds]

    return list(zip(tokens, predicted_tags))

print("\n--- DỰ ĐOÁN THỬ ---")
sample_sentence = "I love learning NLP"
print(predict_sentence(model, sample_sentence, word_to_ix, tag_to_ix, device))

Bắt đầu huấn luyện trên cpu...
Epoch: 01
	Train Loss: 0.607
	 Val. Loss: 0.462 |  Val. Acc: 85.91%
Epoch: 02
	Train Loss: 0.258
	 Val. Loss: 0.401 |  Val. Acc: 87.78%
Epoch: 03
	Train Loss: 0.169
	 Val. Loss: 0.442 |  Val. Acc: 88.10%
Epoch: 04
	Train Loss: 0.138
	 Val. Loss: 0.491 |  Val. Acc: 88.21%
Epoch: 05
	Train Loss: 0.127
	 Val. Loss: 0.506 |  Val. Acc: 88.23%
Epoch: 06
	Train Loss: 0.124
	 Val. Loss: 0.574 |  Val. Acc: 88.16%
Epoch: 07
	Train Loss: 0.121
	 Val. Loss: 0.534 |  Val. Acc: 88.27%
Epoch: 08
	Train Loss: 0.123
	 Val. Loss: 0.616 |  Val. Acc: 88.17%
Epoch: 09
	Train Loss: 0.120
	 Val. Loss: 0.614 |  Val. Acc: 87.92%
Epoch: 10
	Train Loss: 0.122
	 Val. Loss: 0.649 |  Val. Acc: 87.74%

--- DỰ ĐOÁN THỬ ---
[('I', 'PRON'), ('love', 'VERB'), ('learning', 'VERB'), ('NLP', 'NOUN')]


In [44]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx, dropout_rate=0.5):
        super().__init__()

        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # 2. LSTM Layer (Thay cho RNN)
        # batch_first=True: input shape (Batch, Seq, Dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        # 3. Dropout Layer (QUAN TRỌNG: Giúp giảm Overfitting)
        self.dropout = nn.Dropout(dropout_rate)

        # 4. Linear Layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: [batch, seq_len]

        # Qua Embedding và Dropout
        embedded = self.dropout(self.embedding(x))

        # Qua LSTM
        # LSTM trả về (output, (hidden, cell)), ta chỉ cần output
        outputs, _ = self.lstm(embedded)

        # Qua thêm 1 lớp Dropout nữa trước khi vào FC (tùy chọn, để train kỹ hơn)
        outputs = self.dropout(outputs)

        # Qua Linear để dự đoán
        predictions = self.fc(outputs)

        return predictions

In [45]:
from sklearn.metrics import classification_report
import numpy as np

def get_detailed_report(model, iterator, tag_to_ix):
    model.eval()

    # Danh sách chứa tất cả nhãn thật và dự đoán (đã bỏ padding)
    all_preds = []
    all_true = []

    # Map index ngược lại thành text (0 -> PAD, 1 -> NOUN...)
    ix_to_tag = {v: k for k, v in tag_to_ix.items()}

    with torch.no_grad():
        for batch in iterator:
            text, tags = batch
            text = text.to(device)
            tags = tags.to(device)

            predictions = model(text)
            preds = torch.argmax(predictions, dim=-1)

            # Phẳng hóa (Flatten) để duyệt qua từng token
            preds = preds.view(-1).cpu().numpy()
            tags = tags.view(-1).cpu().numpy()

            # Lọc bỏ Padding (QUAN TRỌNG)
            for p, t in zip(preds, tags):
                if t != PAD_IDX: # Chỉ lấy nếu nhãn thật không phải PAD
                    all_preds.append(ix_to_tag[p])
                    all_true.append(ix_to_tag[t])

    # Trả về báo cáo dạng text
    return classification_report(all_true, all_preds, digits=4)

In [46]:
import time

# --- CẤU HÌNH ---
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = len(tag_to_ix)
INPUT_DIM = len(word_to_ix)
N_EPOCHS = 15           # Tăng epoch lên vì có Dropout nên model học chậm hơn nhưng chắc hơn
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.5      # Tỷ lệ tắt neuron ngẫu nhiên

# 1. Khởi tạo Model mới
model = LSTMTagger(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, PAD_IDX, DROPOUT_RATE)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Biến để lưu model tốt nhất
best_valid_loss = float('inf')

print(f"Bắt đầu huấn luyện LSTM (Dropout={DROPOUT_RATE})...")
print("-" * 60)

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, dev_loader, criterion)

    # --- CƠ CHẾ LƯU MODEL TỐT NHẤT ---
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pt') # Lưu trọng số vào file
        saved_msg = "--> Đã lưu Best Model!"
    else:
        saved_msg = ""

    end_time = time.time()
    mins, secs = divmod(end_time - start_time, 60)

    print(f'Epoch: {epoch+1:02} | Time: {int(mins)}m {int(secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}% {saved_msg}')

print("-" * 60)
print("Đã xong! Đang tải lại model tốt nhất để đánh giá...")

# 2. Load lại model tốt nhất để đánh giá (tránh lấy model ở epoch cuối bị overfit)
model.load_state_dict(torch.load('best_model.pt'))

# 3. In báo cáo chi tiết
print("\n=== BÁO CÁO CHI TIẾT TRÊN TẬP DEV (VALIDATION) ===")
report = get_detailed_report(model, dev_loader, tag_to_ix)
print(report)

Bắt đầu huấn luyện LSTM (Dropout=0.5)...
------------------------------------------------------------
Epoch: 01 | Time: 0m 29s
	Train Loss: 1.559
	 Val. Loss: 0.995 |  Val. Acc: 67.70% --> Đã lưu Best Model!
Epoch: 02 | Time: 0m 27s
	Train Loss: 1.027
	 Val. Loss: 0.800 |  Val. Acc: 73.10% --> Đã lưu Best Model!
Epoch: 03 | Time: 0m 31s
	Train Loss: 0.869
	 Val. Loss: 0.685 |  Val. Acc: 77.10% --> Đã lưu Best Model!
Epoch: 04 | Time: 0m 33s
	Train Loss: 0.772
	 Val. Loss: 0.624 |  Val. Acc: 78.91% --> Đã lưu Best Model!
Epoch: 05 | Time: 0m 34s
	Train Loss: 0.697
	 Val. Loss: 0.574 |  Val. Acc: 80.56% --> Đã lưu Best Model!
Epoch: 06 | Time: 0m 31s
	Train Loss: 0.639
	 Val. Loss: 0.531 |  Val. Acc: 82.53% --> Đã lưu Best Model!
Epoch: 07 | Time: 0m 32s
	Train Loss: 0.590
	 Val. Loss: 0.505 |  Val. Acc: 83.61% --> Đã lưu Best Model!
Epoch: 08 | Time: 0m 31s
	Train Loss: 0.549
	 Val. Loss: 0.482 |  Val. Acc: 84.36% --> Đã lưu Best Model!
Epoch: 09 | Time: 0m 32s
	Train Loss: 0.515
	 Val.